In [1]:
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'module_2/week_6/use_case_1'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2101, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 2101 (delta 11), reused 15 (delta 3), pack-reused 2066 (from 1)
Receiving objects: 100% (2101/2101), 263.06 MiB | 20.30 MiB/s, done.
Resolving deltas: 100% (400/400), done.
Updating files: 100% (1348/1348), done.
Working directory: /content/BITS_programming/module_2/week_6/use_case_1


In [2]:
!pip install -q awscli
import os
import json

# ==========================================
# 1. LOAD AWS CREDENTIALS FROM COLAB SECRETS
# ==========================================
def get_colab_secret(name, required=True):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and (value is None or value == ""):
        raise RuntimeError(f"Add the Colab Secret {name} and grant this notebook access.")
    return value

AWS_ACCESS_KEY_ID = get_colab_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = get_colab_secret("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = get_colab_secret("AWS_SESSION_TOKEN", required=False)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN

_aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "us-east-1"
os.environ["AWS_REGION"] = _aws_region
os.environ["AWS_DEFAULT_REGION"] = _aws_region
os.environ["AWS_DEFAULT_OUTPUT"] = "json"

print(f"AWS credentials loaded. Region: {_aws_region}")

# ==========================================
# 2. RUN NON-INTERACTIVE IAM ROLE AUTOMATION
# ==========================================
!echo "--- 1. Testing AWS Credentials ---"
!aws sts get-caller-identity

!echo "--- 2. Generating Temporary SageMaker Trust Policy ---"
trust_policy = {
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": { "Service": "sagemaker.amazonaws.com" },
      "Action": "sts:AssumeRole"
    }
  ]
}

with open("sagemaker-trust-policy.json", "w") as f:
    json.dump(trust_policy, f)

!echo "--- 3. Creating Custom IAM Roles ---"
!aws iam create-role --role-name Lesson6CleanExecRole --assume-role-policy-document file://sagemaker-trust-policy.json || true
!aws iam create-role --role-name model-engineering-lab-sagemaker-execution --assume-role-policy-document file://sagemaker-trust-policy.json || true
!aws iam create-role --role-name SageMakerNeoLabRole --assume-role-policy-document file://sagemaker-trust-policy.json || true
!aws iam create-role --role-name unified-mlops-mlflow-dev-execution-role --assume-role-policy-document file://sagemaker-trust-policy.json || true

!echo "--- 4. Attaching AWS Managed Policies ---"
!aws iam attach-role-policy --role-name Lesson6CleanExecRole --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess
!aws iam attach-role-policy --role-name model-engineering-lab-sagemaker-execution --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess
!aws iam attach-role-policy --role-name SageMakerNeoLabRole --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess

# Attach both SageMaker and S3 permissions for the MLOps dev role
!aws iam attach-role-policy --role-name unified-mlops-mlflow-dev-execution-role --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess
!aws iam attach-role-policy --role-name unified-mlops-mlflow-dev-execution-role --policy-arn arn:aws:iam::aws:policy/AmazonS3FullAccess

!echo "--- 5. Cleanup Temporary Files ---"
!rm -f sagemaker-trust-policy.json

!echo "--- 6. Verification ---"
!aws iam get-role --role-name unified-mlops-mlflow-dev-execution-role --query "Role.[RoleName, Arn]" --output text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 27.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
AWS credentials loaded. Region: us-east-1
--- 1. Testing AWS Credentials ---
{
    "UserId": "455865672536",
    "Account": "455865672536",
    "Arn": "arn:aws:iam::455865672536:root"
}
--- 2. Generating Temporary SageMaker Trust Policy ---
--- 3. Creating Custom IAM Roles ---

An error occurred (EntityAlreadyExists) when calling the CreateRole operation: Role with name Lesson6CleanExecRole already exists.

An error occurred (EntityAlreadyExists) when calling the CreateRole operation: Role with name model-engineering-lab-sagemaker-execution already exists.

An erro

In [3]:
# Create the S3 buckets this notebook uses (idempotent - only creates missing ones).
!pip install -q boto3
import boto3
from botocore.exceptions import ClientError
import os

# 1. Initialize S3 & STS clients using your loaded region
_region = os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or "us-east-1"
_s3 = boto3.client("s3", region_name=_region)
_sts = boto3.client("sts", region_name=_region)

# 2. Get your AWS Account ID to ensure global bucket uniqueness
try:
    account_id = _sts.get_caller_identity()["Account"]
except Exception as e:
    raise RuntimeError(f"Failed to authenticate with AWS credentials: {e}")

# 3. Define globally unique bucket names (Appends Account ID)
BASE_BUCKETS = ['usecase-etl-1']
REQUIRED_BUCKETS = [f"{b}-{account_id}" for b in BASE_BUCKETS]

CREATED_BUCKETS = []

for _b in REQUIRED_BUCKETS:
    bucket_successfully_created_or_verified = False
    try:
        print(f"Attempting to create globally unique bucket: {_b} in region: {_region}")
        if _region != "us-east-1":
            _s3.create_bucket(
                Bucket=_b,
                CreateBucketConfiguration={'LocationConstraint': _region}
            )
        else:
            _s3.create_bucket(Bucket=_b)

        CREATED_BUCKETS.append(_b)
        print(f"✅ Successfully created bucket: {_b}")
        bucket_successfully_created_or_verified = True

    except ClientError as _e:
        _code = _e.response.get("Error", {}).get("Code", "")
        if _code in ("BucketAlreadyOwnedByYou", "Conflict"):
            print(f"ℹ️ Bucket {_b} already owned by your account. Verifying accessibility...")
            try:
                _s3.head_bucket(Bucket=_b)
                print(f"✅ Bucket {_b} verified as accessible.")
                bucket_successfully_created_or_verified = True
            except ClientError as head_e:
                print(f"❌ ERROR accessing existing bucket {_b}: {head_e}")
        elif _code == "BucketAlreadyExists":
            print(f"❌ FATAL: Name collision! Bucket {_b} is globally owned by another AWS user.")
        else:
            print(f"❌ Could not create bucket {_b} ({_code}): {_e}")

    if not bucket_successfully_created_or_verified:
        raise RuntimeError(f"FATAL: S3 bucket {_b} is not ready. Please check AWS credentials, region, and bucket permissions.")

print("Buckets ready:", REQUIRED_BUCKETS)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.6 MB/s eta 0:00:00
Attempting to create globally unique bucket: usecase-etl-1-455865672536 in region: us-east-1
✅ Successfully created bucket: usecase-etl-1-455865672536
Buckets ready: ['usecase-etl-1-455865672536']


In [4]:
# Bucket used by this notebook's S3 reads/writes (same as Notebook_2 - account-suffixed).
S3_BUCKET = REQUIRED_BUCKETS[0]

INPUT_S3_URI  = f"s3://{S3_BUCKET}/raw/online_retail_sample.csv"
OUTPUT_S3_URI = f"s3://{S3_BUCKET}/processed/retail_exploration_ready.csv"

print('Using bucket   :', S3_BUCKET)
print('Input source   :', INPUT_S3_URI)
print('Output target  :', OUTPUT_S3_URI)

Using bucket   : usecase-etl-1-455865672536
Input source   : s3://usecase-etl-1-455865672536/raw/online_retail_sample.csv
Output target  : s3://usecase-etl-1-455865672536/processed/retail_exploration_ready.csv


## Dependencies and files used

### Python packages
- **pathlib** — used to build file paths clearly and safely
- **pandas** — used because it is the most learner-friendly way to inspect CSV data row by row and column by column

### Input file
- `online_retail_sample.csv`

### Output file
- `retail_exploration_ready.csv`

### Why this notebook comes first
This notebook is intentionally lightweight. It helps learners first understand the incoming data before we move to cleaning and ETL automation.

In [5]:
# Standard library import: Path makes file handling readable and OS-independent.
from pathlib import Path

# pandas is used for interactive inspection of CSV data.
import pandas as pd



## Step 1 — Load the raw CSV

### Why this step is performed
Before cleaning or transforming anything, we load the source file exactly as received. This helps learners see the **true incoming structure** rather than a pre-cleaned version.

### Expected result
A pandas DataFrame containing the retail transactions.

In [6]:
# Read the source CSV from the shared S3 bucket, falling back to the local copy.
try:
    df = pd.read_csv(f"s3://{S3_BUCKET}/raw/online_retail_sample.csv")
    print(f"Read from S3: s3://{S3_BUCKET}/raw/online_retail_sample.csv")
except Exception as _e:
    print(f"S3 read failed ({_e}); reading local file instead.")
    df = pd.read_csv("online_retail_sample1.csv")
# Display the first few rows so learners can connect column names to business meaning.
print('Shape:', df.shape)
display(df.head())

S3 read failed (Install s3fs to access S3); reading local file instead.
Shape: (520, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,71053,WHITE METAL LANTERN,6,02/01/2011 11:08,5.49,17889.0,Belgium
1,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,2,01/28/2011 11:32,4.22,16943.0,Germany
2,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,02/05/2011 08:48,5.80,18065.0,Netherlands
3,536366,22752,SET 7 BABUSHKA NESTING BOXES,4,01/13/2011 13:54,7.55,14512.0,United Kingdom
4,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,03/22/2011 17:56,4.03,17075.0,Germany


## Step 2 — Inspect structure and schema

### Why this step is performed
A data engineer first checks whether columns, data types, and row counts look reasonable.
This is where we start asking questions such as:
- Is `InvoiceDate` still a string?
- Is `CustomerID` complete?
- Are numeric fields stored correctly?

### Expected result
A first technical understanding of the dataset schema.

In [7]:
# info() shows column names, non-null counts, and data types.
# This helps us understand how much cleaning may be needed later.
print(df.info())

# Show column-by-column data types in a compact way.
display(pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str)}))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    520 non-null    int64  
 1   StockCode    520 non-null    object 
 2   Description  506 non-null    object 
 3   Quantity     520 non-null    int64  
 4   InvoiceDate  520 non-null    object 
 5   UnitPrice    520 non-null    float64
 6   CustomerID   508 non-null    float64
 7   Country      520 non-null    object 
dtypes: float64(2), int64(2), object(4)
memory usage: 32.6+ KB
None


,column,dtype
InvoiceNo,InvoiceNo,int64
StockCode,StockCode,object
Description,Description,object
Quantity,Quantity,int64
InvoiceDate,InvoiceDate,object
UnitPrice,UnitPrice,float64
CustomerID,CustomerID,float64
Country,Country,object


## Step 3 — Run quality checks

### Why this step is performed
These checks tell us **what kind of transformation work is required next**.
For example:
- missing `CustomerID` may affect customer-level analytics
- missing `Description` may break category derivation
- duplicate rows may overstate revenue
- negative quantity may represent returns
- zero or negative price may signal invalid sales records

### Expected result
A compact quality summary that can be explained to learners.

In [8]:
# Count key quality issues that we want learners to spot early.
quality_summary = pd.DataFrame([
    {'check': 'Missing CustomerID', 'count': int(df['CustomerID'].isna().sum())},
    {'check': 'Missing Description', 'count': int(df['Description'].isna().sum())},
    {'check': 'Duplicate rows', 'count': int(df.duplicated().sum())},
    {'check': 'Negative Quantity', 'count': int((df['Quantity'] < 0).sum())},
    {'check': 'Zero/Negative UnitPrice', 'count': int((df['UnitPrice'] <= 0).sum())},
])

# Show the summary and a few suspicious rows learners can discuss.
display(quality_summary)

suspicious_rows = df[(df['Quantity'] < 0) | (df['UnitPrice'] <= 0) | (df['CustomerID'].isna()) | (df['Description'].isna())]
display(suspicious_rows.head(10))

,check,count
0,Missing CustomerID,12
1,Missing Description,14
2,Duplicate rows,20
3,Negative Quantity,7
4,Zero/Negative UnitPrice,11


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
30,536375,22752,NaN,4,03/11/2011 12:39,3.30,17703.0,Spain
33,536376,85123A,WHITE HANGING HEART T-LIGHT HOLDER,3,03/23/2011 12:10,-2.38,16865.0,Belgium
36,536377,21730,NaN,12,01/20/2011 11:55,4.92,13796.0,Netherlands
38,536378,22633,NaN,24,02/28/2011 13:19,1.84,14210.0,Germany
46,536380,84879,NaN,24,01/31/2011 18:19,3.58,13977.0,Germany
54,536383,84879,ASSORTED COLOUR BIRD ORNAMENT,1,03/15/2011 14:30,6.56,NaN,Netherlands
83,536393,84879,ASSORTED COLOUR BIRD ORNAMENT,1,03/28/2011 12:36,3.46,NaN,Spain
125,536407,71053,WHITE METAL LANTERN,1,02/17/2011 10:35,8.88,NaN,Belgium
133,536409,84879,NaN,12,02/09/2011 08:14,5.45,17255.0,United Kingdom
135,536410,22632,HAND WARMER RED POLKA DOT,6,03/13/2011 18:51,-7.60,15598.0,Belgium


## Step 4 — Parse dates and perform business exploration

### Why this step is performed
The business usually asks questions in terms of **time, geography, and customers**.
So we parse the date field and explore:
- how many invoices we have
- how many customers appear
- what countries exist
- what the date range looks like

### Expected result
A business-readable summary that can be connected to the lab story.

In [9]:
# Convert the date column into a true datetime field.
# errors='coerce' converts invalid date values to NaT so we can detect them.
df['InvoiceDateParsed'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

exploration_summary = pd.DataFrame([
    {'metric': 'Rows', 'value': int(len(df))},
    {'metric': 'Columns', 'value': int(len(df.columns))},
    {'metric': 'Unique invoices', 'value': int(df['InvoiceNo'].nunique())},
    {'metric': 'Unique customers', 'value': int(df['CustomerID'].nunique())},
    {'metric': 'Countries', 'value': int(df['Country'].nunique())},
    {'metric': 'Invalid parsed dates', 'value': int(df['InvoiceDateParsed'].isna().sum())},
])

display(exploration_summary)

display(df['Country'].value_counts().rename_axis('Country').reset_index(name='rows'))

,metric,value
0,Rows,520
1,Columns,9
2,Unique invoices,167
3,Unique customers,465
4,Countries,6
5,Invalid parsed dates,0


,Country,rows
0,France,95
1,United Kingdom,89
2,Belgium,87
3,Spain,86
4,Netherlands,83
5,Germany,80


## Optional analysis — Change one value and observe how the quality result changes

### Why this is useful for learners
A good data engineer does not just run code; they understand how data decisions affect downstream logic.
This small scenario demonstrates that when a price turns invalid, the quality summary changes immediately.

### Scenario
We simulate a bad input by forcing one valid `UnitPrice` value to `0`.
Then we compare the suspicious price count **before vs after**.

In [10]:
scenario_df = df.copy()
original_bad_price_count = int((df['UnitPrice'] <= 0).sum())

# Simulate a bad input record.
scenario_df.loc[0, 'UnitPrice'] = 0
new_bad_price_count = int((scenario_df['UnitPrice'] <= 0).sum())

scenario_result = pd.DataFrame([
    {'metric': 'Original suspicious price count', 'value': original_bad_price_count},
    {'metric': 'After changing one price to 0', 'value': new_bad_price_count},
    {'metric': 'Impact on quality checks', 'value': new_bad_price_count - original_bad_price_count},
])

display(scenario_result)

,metric,value
0,Original suspicious price count,11
1,After changing one price to 0,12
2,Impact on quality checks,1


## Step 5 — Create the exploration-ready output

### Why this step is performed
We now create a lightly standardized file for the next use case.
This is **not full cleaning yet**. The goal is simply to prepare the file so Notebook 2 can continue the flow.

### Standardization done here
- fill missing descriptions with `UNKNOWN_ITEM`
- fill missing customers with `GUEST`
- drop exact duplicate rows

### Expected result
A saved file called `retail_exploration_ready.csv`.

In [11]:
!pip install boto3 s3fs
prepared = df.copy()

# Fill text and identifier gaps with simple placeholders for the next stage.
prepared['Description'] = prepared['Description'].fillna('UNKNOWN_ITEM')
prepared['CustomerID'] = prepared['CustomerID'].fillna('GUEST')

# Remove exact duplicates so downstream counts are more stable.
prepared = prepared.drop_duplicates().reset_index(drop=True)

# Keep the original invoice date string and the parsed version for transparency.
OUTPUT_S3_URI = f"s3://{S3_BUCKET}/processed/retail_exploration_ready.csv"
prepared.to_csv(OUTPUT_S3_URI, index=False)
print('Saved prepared dataset to:', OUTPUT_S3_URI)


print('Prepared shape:', prepared.shape)
display(prepared.head())

INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/

/usr/local/lib/python3.13/dist-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


Saved prepared dataset to: s3://usecase-etl-1-455865672536/processed/retail_exploration_ready.csv
Prepared shape: (500, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDateParsed
0,536365,71053,WHITE METAL LANTERN,6,02/01/2011 11:08,5.49,17889.0,Belgium,2011-02-01 11:08:00
1,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,2,01/28/2011 11:32,4.22,16943.0,Germany,2011-01-28 11:32:00
2,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,02/05/2011 08:48,5.80,18065.0,Netherlands,2011-02-05 08:48:00
3,536366,22752,SET 7 BABUSHKA NESTING BOXES,4,01/13/2011 13:54,7.55,14512.0,United Kingdom,2011-01-13 13:54:00
4,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,03/22/2011 17:56,4.03,17075.0,Germany,2011-03-22 17:56:00


In [12]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-14 08:57:48
